# RAG Data Preparation & Indexing

This notebook prepares a small movie knowledge corpus for Retrieval-Augmented Generation (RAG). It walks through four stages:

1. **Load raw documents** — Define a set of movie-related texts with metadata.
2. **Chunk texts** — Split each document into smaller, overlapping chunks using a recursive character splitter.
3. **Generate embeddings** — Produce dense vector embeddings for each chunk using a SentenceTransformer model.
4. **Persist to Delta** — Save the chunks and embeddings to a Unity Catalog Delta table with Change Data Feed enabled.

## Step 1: Load Raw Documents

Define the catalog/schema context and create a small in-memory corpus of movie-related documents. Each document includes a title, category, and raw text that will be processed downstream.

In [0]:
# --- Unity Catalog target configuration ---
CATALOG = "main"
SCHEMA = "lab_data"

# Set the active catalog/schema so downstream table operations resolve correctly
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

# --- Define the raw knowledge corpus ---
# Each document contains a movie title, a category label, and the full raw text.
# This in-memory corpus simulates an internal knowledge base that a RAG system
# would retrieve from at query time.
raw_documents = [
    {
        "movie_title": "Inception",
        "category": "Production & Plot",
        "raw_text": """Inception is a 2010 science fiction film written and directed by Christopher Nolan. 
The plot follows Dom Cobb, a professional thief who steals information by infiltrating the subconscious of his targets through dream-sharing technology. 
Cobb is offered a chance to have his criminal history erased as payment for the implantation of another person's idea into a target's subconscious: a process known as inception. 
The ending remains ambiguous: Cobb spins his totem (a pewter spinning top) to test whether he is in reality or still dreaming. 
Before the top's motion resolves, the screen cuts to black. However, director Nolan revealed in interviews that the presence of Michael Caine's character indicates reality, as Miles only exists in the waking world."""
    },
    {
        "movie_title": "The Dark Knight",
        "category": "Trivia & Cast",
        "raw_text": """The Dark Knight is a 2008 superhero film directed by Christopher Nolan, starring Christian Bale as Bruce Wayne / Batman. 
Heath Ledger posthumously won the Academy Award for Best Supporting Actor for his portrayal of the Joker. 
During the iconic hospital explosion scene, the detonation was delayed momentarily; Ledger remained in character, playing with the detonator until the pyrotechnics resumed. 
The film set a standard for IMAX camera usage in commercial Hollywood blockbusters, shooting 28 minutes of footage with 65mm IMAX cameras."""
    },
    {
        "movie_title": "Interstellar",
        "category": "Scientific Accuracy & Visuals",
        "raw_text": """Interstellar (2014), directed by Christopher Nolan, explores humanity's search for a habitable planet via a wormhole near Saturn. 
Theoretical physicist Kip Thorne served as scientific consultant and executive producer. Thorne provided theoretical equations to the visual effects team at Double Negative (DNEG). 
The simulation of the supermassive black hole Gargantua produced groundbreaking scientific insights into gravitational lensing and accretion disks. 
The time dilation on Miller's planet is extreme: 1 hour spent on the surface equals 7 earth years due to the intense gravitational field."""
    },
    {
        "movie_title": "Avatar",
        "category": "Technology & Box Office",
        "raw_text": """Avatar (2009), written and directed by James Cameron, became the highest-grossing film of all time, surpassing $2.9 billion worldwide. 
The movie pioneered stereoscopic 3D cameras and fusion camera systems developed by Cameron and Vince Pace. 
It heavily relied on photorealistic performance capture technology to portray the Na'vi people and the alien ecosystem of Pandora. 
Cameron waited over a decade to produce the film until computer-generated imagery advanced sufficiently to realize his visual concept."""
    }
]

print(f"Loaded {len(raw_documents)} source knowledge documents.")

## Step 2: Chunk Texts

Split each raw document into smaller, overlapping text chunks using LangChain's `RecursiveCharacterTextSplitter`. Chunking improves retrieval granularity — each chunk is a self-contained piece of information that can be independently embedded and searched.

In [0]:
import uuid
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- Initialize the recursive text splitter ---
# chunk_size=300:  maximum characters per chunk
# chunk_overlap=50: overlap between adjacent chunks to preserve context across boundaries
# separators:      ordered list of split points, tried from most to least specific
#                   (paragraph break → line break → sentence → word)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "]
)

# --- Split each document into chunks and attach metadata ---
# Every chunk inherits its parent document's title and category,
# gets a unique ID (for deduplication / upserts later), and a
# sequential index indicating its position within the source document.
processed_chunks = []

for doc in raw_documents:
    splits = text_splitter.split_text(doc["raw_text"])
    for idx, chunk in enumerate(splits):
        processed_chunks.append({
            "chunk_id": str(uuid.uuid4()),
            "movie_title": doc["movie_title"],
            "category": doc["category"],
            "chunk_index": idx,
            "content": chunk.strip(),
            "char_count": len(chunk.strip())
        })

print(f"Generated {len(processed_chunks)} granular chunks from raw documents.")

## Step 3: Generate Dense Embeddings

Use the `all-MiniLM-L6-v2` SentenceTransformer model to convert each text chunk into a dense vector embedding. These vectors capture semantic meaning and will be used for similarity-based retrieval at query time.

In [0]:
from sentence_transformers import SentenceTransformer

# --- Load the embedding model ---
# all-MiniLM-L6-v2 is a lightweight, fast model that produces 384-dimensional
# dense vectors. It is well-suited for semantic similarity and retrieval tasks.
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# --- Encode all chunk texts into dense vectors ---
# Batch-encode all chunks at once for efficiency, then attach each vector
# back to its corresponding chunk as a list of floats (JSON-serializable).
chunk_texts = [chunk["content"] for chunk in processed_chunks]
embeddings = embedding_model.encode(chunk_texts, show_progress_bar=True)

# Attach embedding vectors to chunk metadata
for i, chunk in enumerate(processed_chunks):
    chunk["embedding"] = embeddings[i].tolist()

print(f"Generated {len(embeddings)} dense embedding vectors (dim={len(embeddings[0])}).")

## Step 4: Persist Chunks & Embeddings to Delta

Write the processed chunks and their embedding vectors to a Unity Catalog Delta table with Change Data Feed enabled. This table serves as the searchable knowledge base for downstream RAG retrieval.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, ArrayType, FloatType

# --- Define the schema for the chunks table ---
# The embedding column is an array of floats matching the model's output dimension (384).
schema = StructType([
    StructField("chunk_id", StringType(), False),
    StructField("movie_title", StringType(), False),
    StructField("category", StringType(), True),
    StructField("chunk_index", IntegerType(), True),
    StructField("content", StringType(), False),
    StructField("char_count", IntegerType(), True),
    StructField("embedding", ArrayType(FloatType()), False)
])

df_chunks = spark.createDataFrame(processed_chunks, schema=schema)

# --- Write to Unity Catalog as a Delta table ---
# Change Data Feed (CDF) is enabled so downstream pipelines can track row-level
# changes (inserts/updates/deletes) for incremental processing.
df_chunks.write.format("delta") \
    .mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(f"{CATALOG}.{SCHEMA}.movie_knowledge_chunks")

print(f"Successfully written chunks and embeddings to `{CATALOG}.{SCHEMA}.movie_knowledge_chunks`.")

# --- Verify: preview the first 5 rows (embedding column omitted for readability) ---
display(spark.sql(f"SELECT chunk_id, movie_title, category, chunk_index, content, char_count FROM {CATALOG}.{SCHEMA}.movie_knowledge_chunks LIMIT 5"))

## Step 5: Databricks Vector Search Index Setup

Create a native Databricks Vector Search endpoint and Delta Sync Index on the `movie_knowledge_chunks` table for production-grade managed similarity search. This replaces manual pandas cosine similarity with a scalable, serverless search API.

**If the workspace lacks Enterprise Serverless quota**, the notebook gracefully falls back to the in-memory cosine similarity approach used in notebook 02.

**Steps:**
1. Verify data integrity (12 chunks, no null embeddings)
2. Create a Vector Search endpoint
3. Create a Delta Sync Index (self-managed embeddings, dimension 384)

In [0]:
# --- Step 5a: Data Integrity Check ---
chunk_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{SCHEMA}.movie_knowledge_chunks").collect()[0]["cnt"]
null_embeddings = spark.sql(f"SELECT COUNT(*) AS cnt FROM {CATALOG}.{SCHEMA}.movie_knowledge_chunks WHERE embedding IS NULL").collect()[0]["cnt"]

print("📊 Data Integrity Check:")
print(f"  • Total chunks: {chunk_count}")
print(f"  • Null embeddings: {null_embeddings}")

if chunk_count != 12:
    print(f"⚠️ Warning: Expected 12 chunks, found {chunk_count}.")
elif null_embeddings > 0:
    print(f"⚠️ Warning: {null_embeddings} chunks have null embeddings.")
else:
    print("✅ All 12 chunks present with valid non-null embeddings.")

# --- Step 5b: Vector Search Endpoint & Delta Sync Index ---
ENDPOINT_NAME = "movie_rag_endpoint"
INDEX_NAME = f"{CATALOG}.{SCHEMA}.movie_knowledge_index"
SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.movie_knowledge_chunks"

try:
    # In-block import for safe interception in the Free Edition
    from databricks.vector_search.client import VectorSearchClient
    
    vsc = VectorSearchClient()

    # Attempt to retrieve or create a Vector Search Endpoint
    try:
        endpoint = vsc.get_endpoint(ENDPOINT_NAME)
        print(f"Endpoint '{ENDPOINT_NAME}' already exists.")
    except Exception:
        print(f"Creating endpoint '{ENDPOINT_NAME}'...")
        vsc.create_endpoint(name=ENDPOINT_NAME, endpoint_type="STANDARD")
        print(f"Endpoint '{ENDPOINT_NAME}' created successfully.")

    # Attempt to create or retrieve the Delta Sync Index
    try:
        index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)
        print(f"Index '{INDEX_NAME}' already exists.")
    except Exception:
        print(f"Creating Delta Sync Index '{INDEX_NAME}'...")
        vsc.create_delta_sync_index(
            endpoint_name=ENDPOINT_NAME,
            index_name=INDEX_NAME,
            source_table_name=SOURCE_TABLE,
            pipeline_type="TRIGGERED",
            primary_key="chunk_id",
            embedding_vector_column="embedding",
            embedding_dimension=384
        )
        print(f"Index '{INDEX_NAME}' created and syncing.")

    print("\n Databricks Vector Search is ready for production-grade similarity search.")

except (ImportError, ModuleNotFoundError):
    print("\nℹ️ 'databricks-vectorsearch' client not installed in this environment.")
    print("✅ Graceful Fallback: Vector Search Endpoint requires Enterprise Serverless quota.")
    print("   In-memory Cosine Similarity (Notebook 02) remains the active retrieval engine.")

except Exception as e:
    print(f"\nℹ️ Vector Search setup info: {e}")
    print("✅ Graceful Fallback: In-memory Cosine Similarity (Notebook 02) remains the active retrieval engine.")